# Thermal-to-Depth Generation Notebook

This notebook converts the selected thermal input images into depth maps for use as structural conditioning signals. The thermal images are loaded, converted into the required RGB format for the depth estimation model, and processed through a pretrained depth estimation pipeline. The resulting depth maps are saved for later use in both the baseline ControlNet pipeline and the TADN-enhanced pipeline.

In [15]:
import cv2
import numpy as np
from PIL import Image
from pathlib import Path

In [17]:
ROOT = Path.cwd()
INPUT_DIR = ROOT / "input (thermal)"
OUT_DIR = ROOT / "depth outputs"
OUT_DIR.mkdir(exist_ok=True)

print("Input:", INPUT_DIR)
print("Output:", OUT_DIR)

Input: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input (thermal)
Output: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth outputs


In [19]:
%pip install -q torch torchvision torchaudio
%pip install -q opencv-python pillow matplotlib timm einops

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [20]:
# Load thermal image (grayscale)
img_gray = cv2.imread(str(INPUT_DIR / "t1.jpeg"), cv2.IMREAD_GRAYSCALE)

if img_gray is None:
    raise FileNotFoundError("Could not load t1.jpeg")

print("Thermal shape:", img_gray.shape, img_gray.dtype)

# Convert to RGB ONLY because Depth Anything expects RGB
img_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)

pil_img = Image.fromarray(img_rgb)

Thermal shape: (512, 640) uint8


In [21]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def load_thermal(path: Path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img

In [22]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

from transformers import pipeline
print("pipeline import OK")

torch: 2.1.2
transformers: 4.36.2
pipeline import OK


In [23]:
from transformers import pipeline
import numpy as np
from PIL import Image

# Create depth estimation pipeline
depth_pipe = pipeline(
    task="depth-estimation",
    model="Intel/dpt-large"
)

print("Depth pipeline ready")

/opt/homebrew/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DPTForDepthEstimation were not initialized from the model checkpoint at Intel/dpt-large and are newly initialized: ['neck.fusion_stage.layers.0.residual_layer1.convolution1.bias', 'neck.fusion_stage.layers.0.residual_layer1.convolution1.weight', 'neck.fusion_stage.layers.0.residual_layer1.convolution2.bias', 'neck.fusion_stage.layers.0.residual_layer1.convolution2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration.


Depth pipeline ready


In [ ]:
import torch
import numpy as np
from PIL import Image
from pathlib import Path
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

ROOT = Path("/Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox")
IN_PATH = ROOT / "input (thermal)" / "t1.jpeg"
OUT_DIR = ROOT / "depth outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

img = Image.open(IN_PATH).convert("RGB")

MODEL_ID = "Intel/dpt-hybrid-midas"
DEVICE = "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

with torch.no_grad():
    inputs = processor(images=img, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    outputs = model(**inputs)
    pred = outputs.predicted_depth  

    pred = torch.nn.functional.interpolate(
        pred.unsqueeze(1),
        size=img.size[::-1],  
        mode="bicubic",
        align_corners=False,
    ).squeeze()

depth = pred.cpu().numpy()
depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
depth_u8 = (depth * 255).astype(np.uint8)

out_path = OUT_DIR / "t1_depth.png"
Image.fromarray(depth_u8).save(out_path)
print("Saved:", out_path)

/Users/maryamellathy/Desktop/NEW thermaltoRBG/.venv_color/lib/python3.11/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W0128 16:59:27.069000 72634 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
/Users/maryamellathy/Desktop/NEW thermaltoRBG/.venv_color/lib/python3.11/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/Users/maryamellathy/Desktop/NEW thermaltoRBG/.venv_color/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If 

✅ Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth outputs/t1_depth.png


In [7]:
import cv2
import numpy as np
from PIL import Image
from pathlib import Path

INPUT_DIR = ROOT / "input (thermal)"

for i in range(1, 10):
    img_gray = cv2.imread(str(INPUT_DIR / f"t{i}.jpeg"), cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        print(f"t{i}.jpeg not found, skipping")
        continue

    img_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
    pil_img = Image.fromarray(img_rgb)

In [ ]:
from PIL import Image

i = 1
IN_PATH = ROOT / "input (thermal)" / "t1.jpeg"
control_path = ROOT / "t1_depth_tadn_safe.jpeg"

init_rgb = Image.open(IN_PATH).convert("RGB")
control_safe = Image.open(control_path).convert("RGB")

target_size = (512, 512)
init_rgb = init_rgb.resize(target_size)
control_safe = control_safe.resize(target_size)

print("init_rgb:", init_rgb.size)
print("control_safe:", control_safe.size)